In [ ]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

from tqdm import tqdm 

from loka.data import load_breakthrough_states
from loka.models import BreakthroughCNN
from loka.training_methods import ForgettingScores

import os
import pandas as pd
from dotenv import load_dotenv

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

load_dotenv()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
BASE = os.getenv("BASE_DIR")

X,Y = load_breakthrough_states(f"{BASE}/data/game_states/e.txt")
print(X.shape)

A forgetting event is when
\begin{equation}
\mathbf{1}_{(y^t_i = y_i)} = 1 > \mathbf{1}_{(y^{t+1}_i = y_i)} = 0
\end{equation}

This is calculated with
\begin{equation} C^{[t+1]}=C^{[t]}+\max\left(\mathbf{1}(Y=M^{[t]}(X))-\mathbf{1}(Y=M^{[t+1]}(X)),\mathbf{0}\right) \end{equation}

In [ ]:
start = 0.01
end = 0.1
steps = 50
batch_size = 512

splits = np.linspace(start, end, num=steps, endpoint=True)
mse_values = []
prc_values = []

X_t = torch.FloatTensor(X).to(device)
Y_t = torch.FloatTensor(Y).to(device)

# Setup forgetting count
C = torch.zeros(len(X_t), dtype=torch.int32, device=device)
base = torch.zeros(len(X_t),dtype= torch.int32, device=device)
prev_correct = torch.zeros(len(X_t), dtype=torch.int32, device=device)

epochs = 50
pbar = tqdm(total=steps * epochs, desc='training')

for seed, ratio in enumerate(splits):
    X_train, Y_train = X[:int(ratio * len(X))], Y[:int(ratio * len(X))] 

    torch.manual_seed(SEED)  # same init weights each time
    model = BreakthroughCNN().to(device)

    # training setup
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()
    
    # convert data to tensors
    X_train_t = torch.FloatTensor(X_train).to(device)
    Y_train_t = torch.FloatTensor(Y_train).to(device)
    
    # training loop
    for epoch in range(epochs):
        perm = torch.randperm(len(X_train_t), device=device)  # new shuffle each epoch
        
        for i in range(0, len(X_train_t), batch_size):
            idx = perm[i:i + batch_size]
            xb, yb = X_train_t[idx], Y_train_t[idx]
    
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        pbar.update(1)
    
    # evaluate
    with torch.no_grad():
        test_preds = model(X_t)
        mse = loss_fn(test_preds, Y_t).item()
        mse_values.append(mse)
        prc_values.append(ratio)

    # inside the loop, replacing the evaluate + forgetting section
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_t), batch_size):
            preds.append(model(X_t[i:i+batch_size]))
    test_preds = torch.cat(preds)
    mse = loss_fn(test_preds, Y_t).item()
    mse_values.append(mse)
    prc_values.append(ratio)
    
    # inside the loop
    correct = (test_preds.clamp(0, 1).round() == Y_t).int()   # 0 or 1
    
    # forgetting event: was correct before, wrong now
    C += torch.maximum(prev_correct-correct, base)
    prev_correct = correct
    

    pbar.set_postfix(model=f'{seed+1}/{steps}', ratio=f'{ratio:.2f}', loss=f'{loss.item():.4f}', mse=f'{mse:.4f}')

    if seed != len(splits) - 1:
        del model, optimizer, X_train_t, Y_train_t
        torch.cuda.empty_cache()

pbar.close()

In [ ]:
import matplotlib.pyplot as plt

counts = torch.bincount(C.cpu())
plt.bar(range(len(counts)), counts.numpy())
plt.xlabel('forgetting events')
plt.ylabel('number of examples')
plt.yscale('log')          # drop this if the bars are comparable
plt.xticks(range(len(counts)))
plt.show()

In [ ]:
start = 0.01
end = 0.1
steps = 50
epochs = 50

splits = np.linspace(start, end, num=steps, endpoint=True)
n_bar_splits = splits / start

pbar = tqdm(
    total=epochs * n_bar_splits.sum(),
    desc="training",
    bar_format="{l_bar}{bar} {percentage:3.0f}% [{elapsed}<{remaining}]",
)

n_bar = 0

all_C = list()
seed = 42


for i,s in enumerate(splits):
    count = int(len(X)*s)
    X_t, Y_t = X[:count], Y[:count]

    torch.manual_seed(seed)
    model = BreakthroughCNN()
    forgetter = ForgettingScores(model, seed=seed)

    forgetter.train(
        X = X_t,
        Y = Y_t,
        epochs = epochs,
        batch_size= 512,
        random_permuations=True,
        device=device,
        pbar=pbar,
        pbar_increment=n_bar_splits[i]
    )

    all_C.append(forgetter.getForgettingCounts())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Data prep ---
Cs = [c.cpu().numpy() for c in all_C]
ns = np.array([len(c) for c in Cs])                       # examples per run
steps = len(Cs)
max_c = int(max(c.max() for c in Cs))

bins = np.arange(max_c + 2) - 0.5                         # one bin per integer count
H = np.stack([np.histogram(c, bins=bins)[0] for c in Cs]) # (steps, max_c + 1) raw counts
H_pct = H / ns[:, None] * 100                             # % of each run's dataset

# --- Style ---
x = np.arange(max_c + 1)
width = 0.9 / steps                                       # thin slice per run
cmap = plt.colormaps["turbo"]
colors = cmap(np.linspace(0, 1, steps))

# --- Plot: linear on top, log below ---
fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

for ax, log in zip(axes, [False, True]):
    for k in range(steps):
        offset = (k - (steps - 1) / 2) * width            # centers the group on the integer
        ax.bar(x + offset, H_pct[k], width=width, color=colors[k], alpha=0.8, linewidth=0)

    if log:
        ax.set_yscale("log")
        ax.set_ylim(100 / ns.max() * 0.5, 100)            # just below one example in the largest run
        ax.set_title("Log scale")
    else:
        ax.set_ylim(0, 100)
        ax.set_title("Linear scale")

    ax.set_ylabel("% of run's dataset")

axes[1].set_xticks(x)
axes[1].set_xlabel("forgetting count")
fig.suptitle("Forgetting count distribution, one slice per run")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(ns.min(), ns.max()))
fig.colorbar(sm, ax=axes.ravel().tolist(), label="number of examples in run")
plt.show()